In [1]:
import json
import numpy as np
import shutil
import pandas as pd

In [2]:
df = pd.read_csv('../mhc_data/TCR3d_data.csv')
df['PDB ID'] = df['PDB ID'].astype(str).str.lower().str.strip()
df['Release date'] = pd.to_datetime(df['Release date'])

In [3]:
mask_train = df['Release date'] < '2021-09-30'
train_df = df[mask_train]

In [4]:
len(train_df)

1082

In [5]:
mask_val = (df['Release date'] >= '2021-09-30') & (df['Release date'] < '2023-01-13')
val_df = df[mask_val]

In [6]:
len(val_df)

130

In [7]:
train_ids = train_df['PDB ID'].dropna().tolist()
val_ids = val_df['PDB ID'].dropna().tolist()

In [ ]:
# Apenas para guardar essa informação, não é necessário rodar esse código novamente
# No resto do código vamos usar o train_ids e val_ids

# with open('../mhc_data/train_templated.txt', 'w') as f:
#     f.write('\n'.join(train_ids))

# with open('../mhc_data/val_templated.txt', 'w') as f:
#     f.write('\n'.join(val_ids))

In [8]:
with open('../rcsb_processed_targets/manifest.json') as f:
    data = json.load(f)

In [9]:
with open('../mhc_data/train_templated.json') as f:
    train = json.load(f)

with open('../mhc_data/val_templated.json') as f:
    val = json.load(f)

In [ ]:
# train_val = [sample for sample in train_val if sample['PDB ID']=='1a1m']

In [24]:
train_val = train + val

In [10]:
manifest_dict = {sample['id']: sample for sample in data}

In [ ]:
import os
import urllib.request

def baixar_cif_do_pdb(pdb_id, diretorio_destino="../cif_files"):
    """
    Baixa o arquivo .cif do banco de dados RCSB PDB.
    """
    os.makedirs(diretorio_destino, exist_ok=True)
    
    # O RCSB PDB fornece arquivos CIF através desta URL padrão
    url = f"https://files.rcsb.org/download/{pdb_id.lower()}.cif"
    caminho_arquivo = os.path.join(diretorio_destino, f"{pdb_id.lower()}.cif")
    
    # Baixa apenas se o arquivo ainda não existir localmente
    if not os.path.exists(caminho_arquivo):
        try:
            urllib.request.urlretrieve(url, caminho_arquivo)
            print(f"Sucesso ao baixar: {pdb_id.upper()}")
        except Exception as e:
            print(f"Erro ao baixar {pdb_id.upper()}: {e}")
            return None
    else:
        print(f"Arquivo já existe: {pdb_id.upper()}")
            
    return caminho_arquivo

with open('../mhc_data/val_templated.txt', 'r') as f:
    trains = [line.strip() for line in f]

print(f"Total de amostras de treino: {len(trains)}")

for i, id in enumerate(trains):
    caminho_cif = baixar_cif_do_pdb(id)
    print(f"{i+1}/{len(trains)} - {id} - {caminho_cif}")
    


In [25]:
target_dir = 'mhc_samples'
OUTPUT_FILE_NAME = 'manifest'

import os
os.makedirs(target_dir, exist_ok=True)
os.makedirs(f"{target_dir}/msa/", exist_ok=True)
os.makedirs(f"{target_dir}/structures/", exist_ok=True)

new_manifest = []

bad_ids = []

for sample in train_val:
    if sample['pdb_id'] not in manifest_dict:
        bad_ids.append(sample['pdb_id'])
        continue
    sample_manifest = manifest_dict[sample['pdb_id']]
    peptide_chain_name = sample['peptide_chain'] + '1'
    protein_chain_name = sample['protein_chains'][0] + '1'
    matched_chains = 0
    valid_chain_ids = []
    for chain in sample_manifest['chains']:
        if chain['msa_id'] != -1:
            shutil.copy(f"../rcsb_processed_msa/{chain['msa_id']}.npz", f"{target_dir}/msa/{chain['msa_id']}.npz")
        if chain['chain_name'] in [peptide_chain_name, protein_chain_name]:
            matched_chains += 1
            valid_chain_ids.append(chain['chain_id'])
        else:
            chain['valid'] = False
    if matched_chains != 2:
        bad_ids.append(sample['pdb_id'])
        print('Didnt find both chains', sample['pdb_id'], matched_chains)
    else:
        n_correct_interfaces = 0
        for interface in sample_manifest['interfaces']:
            if (interface['chain_1'] in valid_chain_ids) and (interface['chain_2'] in valid_chain_ids):
                n_correct_interfaces += 1
            else:
                interface['valid'] = False
        if n_correct_interfaces != 1:
            print('Number of correct interfaces is wrong:', sample['pdb_id'], n_correct_interfaces)
        else:
            new_manifest.append(sample_manifest)
            
            # update mask in npz just in case
            npz = dict(np.load(f"../rcsb_processed_targets/structures/{sample_manifest['id']}.npz"))
            for chain_id in range(len(npz['mask'])):
                if chain_id in valid_chain_ids:
                    npz['mask'][chain_id] = True
                else:
                    npz['mask'][chain_id] = False
            np.savez(f"{target_dir}/structures/{sample_manifest['id']}", **npz)
    # break

Didnt find both chains 1qo3 1
Didnt find both chains 2x4o 1
Didnt find both chains 2x4t 1
Didnt find both chains 2x4u 1
Didnt find both chains 3ch1 0
Didnt find both chains 3tf7 1
Didnt find both chains 4lcy 1
Didnt find both chains 4uq3 1
Didnt find both chains 5ts1 1
Didnt find both chains 5wwj 1
Didnt find both chains 5wxc 1
Didnt find both chains 6gh1 1
Didnt find both chains 6nf7 1
Didnt find both chains 6ss7 1
Didnt find both chains 7k81 0
Didnt find both chains 7n6d 1
Didnt find both chains 7n6e 1
Number of correct interfaces is wrong: 7bh8 0
Didnt find both chains 7l1d 0
Number of correct interfaces is wrong: 7ndt 0
Number of correct interfaces is wrong: 7re8 0


In [27]:
with open(f"{target_dir}/{OUTPUT_FILE_NAME}.json", "w") as outfile:
    outfile.write(json.dumps(new_manifest))

In [22]:
with open(f"{target_dir}/{OUTPUT_FILE_NAME}.txt", "w") as f:
    f.write('\n'.join([sample['id'] for sample in new_manifest]))

In [21]:
len(new_manifest)

1024

In [18]:
with open(f"{target_dir}/bad_ids.txt", "w") as f:
    f.write('\n'.join(bad_ids))

In [ ]:
len(train_val), len(bad_ids)

In [ ]:
val_list = [sample['pdb_id'].upper() for sample in val]

with open("{target_dir}/validation_ids.txt", "w") as outfile:
    outfile.write('\n'.join(val_list))

In [ ]:
# !zip -r mhc.zip mhc_targets/

In [ ]:
# scp mhc.zip eglukhov@nabu5.ams.stonybrook.edu:/home/eglukhov/projects/boltz/train_data/

In [ ]:
< 2021-09-30 - train
< 2023-01-13 - val
> - test